# 03 — Day 3: End-to-End Pipeline Demo

**Goal:** wire the Phase-1 components into `verify_claim()` and produce TRUE / FALSE / ABSTAIN verdicts on 3 curated financial claims.

Pipeline: `claim → decompose (qwen2.5:7b) → retrieve (credibility-weighted, news index) → reason (glm-4.7-flash) → aggregate`. Aggregation rule (Phase-1 placeholder; conformal calibrator is Phase-7): any FALSE proposition → claim FALSE; all TRUE → claim TRUE; mixed/UNCERTAIN → ABSTAIN.

This is the supervisor-pitch demo. Definition of done: each of the 3 claims yields the *expected* verdict with cited evidence.

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
load_dotenv(ROOT / '.env')

True

## Three demo claims

Curated against the `crag_fin_news_v0` index seeded on Day 2 (180 chunks from 2026-05-14 financial news).

1. **Expected TRUE** — *'Amazon stock is approaching a $3 trillion valuation on AI success.'* Bloomberg ran this story today.
2. **Expected FALSE** — *'GameStop bid $100 billion for eBay.'* The actual reported figure (Guardian) is $55.5bn.
3. **Expected ABSTAIN** — *'Apple beat Q3 earnings estimates.'* Our news index has no Apple-Q3 coverage.

In [2]:
from crag_fin.pipeline import verify_claim

demo_claims = [
    'Amazon stock is approaching a $3 trillion valuation on AI success.',
    'GameStop bid $100 billion for eBay.',
    'Apple beat Q3 earnings estimates.',
]

for claim in demo_claims:
    print('=' * 80)
    print(f'CLAIM: {claim}')
    print('=' * 80)
    cv = verify_claim(claim, k=4)
    print(f'VERDICT: {cv.label}')
    for i, t in enumerate(cv.propositions, 1):
        print(f'\n  proposition [{i}]: {t.proposition.proposition}')
        print(f'    type = {t.proposition.type} | entity = {t.proposition.ticker_or_entity}')
        print(f'    verdict = {t.verdict.label}')
        print(f'    justification = {t.verdict.justification[:200]}')
        print(f'    citations = {t.verdict.citations}')
        print('    top retrieved:')
        for j, p in enumerate(t.passages[:3], 1):
            star = '<-' if j in t.verdict.citations else '  '
            print(f'      {star} [{j}] cred={p.credibility:.2f} | {p.source[:55]}')
            print(f'              {p.text[:100]}')
    print()

CLAIM: Amazon stock is approaching a $3 trillion valuation on AI success.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

VERDICT: TRUE

  proposition [1]: Amazon stock is approaching a $3 trillion valuation.
    type = price_movement | entity = AMZN
    verdict = TRUE
    justification = Passage 1 explicitly states that Amazon's market capitalization is "soaring toward the rarefied $3 trillion level." Passages 2, 3, and 4 discuss unrelated topics such as data center electricity usage 
    citations = [1]
    top retrieved:
      <- [1] cred=1.00 | https://www.bloomberg.com/news/articles/2026-05-14/amaz
              Amazon’s AI Success Sends Stock Racing Toward $3 Trillion Club. Investors are growing increasingly o
         [2] cred=0.70 | https://www.theguardian.com/technology/2026/may/13/data
              The proportion of electricity used by vast warehouses stacked with microchips to power AI and the in
         [3] cred=0.70 | https://www.theguardian.com/business/nils-pratley-on-fi
              GameStop hits the limits of credibility with $55.5bn eBay bid | Nils Pratley. The ‘meme stock’ compa

  p

VERDICT: FALSE

  proposition [1]: GameStop bid $100 billion for eBay.
    type = M&A | entity = GME
    verdict = FALSE
    justification = Passages 1 and 2 explicitly state that GameStop's bid for eBay was $55.5 billion, which directly contradicts the claim of a $100 billion bid.
    citations = [1, 2]
    top retrieved:
      <- [1] cred=0.70 | https://www.theguardian.com/business/nils-pratley-on-fi
              GameStop hits the limits of credibility with $55.5bn eBay bid | Nils Pratley. The ‘meme stock’ compa
      <- [2] cred=0.70 | https://www.theguardian.com/business/nils-pratley-on-fi
              It was eBay’s succinct appraisal of the bizarre $55.5bn (£41bn) takeover offer from video games reta
         [3] cred=0.70 | https://www.theguardian.com/business/nils-pratley-on-fi
              Surprisingly, the Redditers succeeded beyond their wildest dreams. The squeeze drove up GameStop’s s

CLAIM: Apple beat Q3 earnings estimates.


VERDICT: ABSTAIN

  proposition [1]: Apple beat Q3 earnings estimates.
    type = earnings | entity = AAPL
    verdict = UNCERTAIN
    justification = The provided passages discuss Versant's Q1 earnings, UK GDP, Cisco's earnings, and an insurance company's Q1 results, none of which mention Apple or its Q3 earnings performance.
    citations = []
    top retrieved:
         [1] cred=0.70 | https://www.cnbc.com/2026/05/14/versant-vsnt-earnings-q
              Versant stock jumps 10% after company's Q1 report shows bright spots in licensing, platforms. Versan
         [2] cred=0.70 | https://www.theguardian.com/business/live/2026/may/14/u
              Speaking to BBC News this morning, the chancellor says: Next week I’ll be setting out more detail on
         [3] cred=1.00 | https://www.bloomberg.com/news/articles/2026-05-14/us-s
              US Stock Futures Gain as AI Trade Drives Tech Rally, Cisco Soars. US stock futures rose Thursday mor



## Day 3 verdict

- The pipeline produces verdicts on real claims against real (today-seeded) news. The 3 demo claims show TRUE / FALSE / ABSTAIN behaviour.
- Threshold abstention (UNCERTAIN → ABSTAIN) is in place; the conformal calibrator replaces it in Phase 7 (Days 19–21).
- README + screencast (next step) make this the supervisor-pitch deliverable. Tag `phase-1-complete` after merging back to `master`.